In [6]:
import anndata as ad

adata = ad.read_h5ad(
    'skin.inflammatory.h5ad'
)
adata

AnnData object with n_obs × n_vars = 332546 × 4000
    obs: 'sample_id', 'patient_id', 'status', 'tissue', 'cell_fraction', 'doublet', 'doublet_score', 'nFeature_RNA', 'nCount_RNA', 'percent_mt', 'percent_ribo', 'qc_pass', '_scvi_batch', '_scvi_labels', 'leiden_scvi_0.1', 'coarse_cell_types', 'leiden_scvi_0.75', 'cell_type'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'cell_type_colors', 'coarse_cell_types_colors', 'hvg', 'leiden', 'leiden_scvi_0.1_colors', 'leiden_scvi_0.75_colors', 'neighbors', 'umap'
    obsm: 'X_scvi', 'X_umap'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [7]:
adata = adata.raw.to_adata()
adata

AnnData object with n_obs × n_vars = 332546 × 20912
    obs: 'sample_id', 'patient_id', 'status', 'tissue', 'cell_fraction', 'doublet', 'doublet_score', 'nFeature_RNA', 'nCount_RNA', 'percent_mt', 'percent_ribo', 'qc_pass', '_scvi_batch', '_scvi_labels', 'leiden_scvi_0.1', 'coarse_cell_types', 'leiden_scvi_0.75', 'cell_type'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'cell_type_colors', 'coarse_cell_types_colors', 'hvg', 'leiden', 'leiden_scvi_0.1_colors', 'leiden_scvi_0.75_colors', 'neighbors', 'umap'
    obsm: 'X_scvi', 'X_umap'
    obsp: 'connectivities', 'distances'

In [8]:
adata.obs.groupby(['patient_id', 'status'], observed = True).count()

,,sample_id,tissue,cell_fraction,doublet,doublet_score,nFeature_RNA,nCount_RNA,percent_mt,percent_ribo,qc_pass,_scvi_batch,_scvi_labels,leiden_scvi_0.1,coarse_cell_types,leiden_scvi_0.75,cell_type
patient_id,status,,,,,,,,,,,,,,,,
P01,sarcoidosis,1587,1587,1587,1587,1587,1587,1587,1587,1587,1587,1587,1587,1587,1587,1587,1587
P1,psoriasis,34740,34740,34740,34740,34740,34740,34740,34740,34740,34740,34740,34740,34740,34740,34740,34740
P02,sarcoidosis,3213,3213,3213,3213,3213,3213,3213,3213,3213,3213,3213,3213,3213,3213,3213,3213
P2,psoriasis,43963,43963,43963,43963,43963,43963,43963,43963,43963,43963,43963,43963,43963,43963,43963,43963
P03,sarcoidosis,1951,1951,1951,1951,1951,1951,1951,1951,1951,1951,1951,1951,1951,1951,1951,1951
P3,psoriasis,40227,40227,40227,40227,40227,40227,40227,40227,40227,40227,40227,40227,40227,40227,40227,40227
P04,sarcoidosis,234,234,234,234,234,234,234,234,234,234,234,234,234,234,234,234
P05,sarcoidosis,2692,2692,2692,2692,2692,2692,2692,2692,2692,2692,2692,2692,2692,2692,2692,2692
P06,sarcoidosis,893,893,893,893,893,893,893,893,893,893,893,893,893,893,893,893


In [9]:
import pandas as pd
import numpy as np


np.random.seed(2026)
subsample = []
cells_per_patient = 2500
for patient_id, cells in adata.obs.groupby('patient_id'):
    if len(cells) < cells_per_patient:
        subsample.append(cells)
        continue
        
    bycells = []
    for celltype, group in cells.groupby('cell_type', observed = True):
        nsample = int(cells_per_patient * len(group) / len(cells))
        bycells.append(group.sample(nsample))

    subsample.append(pd.concat(bycells))

samples = pd.concat(subsample)
samples

/tmp/ipykernel_2496177/1730957851.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for patient_id, cells in adata.obs.groupby('patient_id'):


,sample_id,patient_id,status,tissue,cell_fraction,doublet,doublet_score,nFeature_RNA,nCount_RNA,percent_mt,percent_ribo,qc_pass,_scvi_batch,_scvi_labels,leiden_scvi_0.1,coarse_cell_types,leiden_scvi_0.75,cell_type
AAACCCAAGCCTCACG-11,P01_V02_Skin_L,P01,sarcoidosis,Skin_L,None,False,1.722537,3458,11449.0,5.694821,9.564154,True,24,0,1,Tcell,14,Treg_SAT1_hi
AAACCCATCAGCGCGT-11,P01_V02_Skin_L,P01,sarcoidosis,Skin_L,None,False,3.348081,4701,21711.0,11.312238,10.672010,True,24,0,2,other,10,Macro_2
AAACGAATCGCGCTGA-11,P01_V02_Skin_L,P01,sarcoidosis,Skin_L,None,False,1.047968,3316,11736.0,10.702113,10.838446,True,24,0,2,other,10,Macro_2
AAACGCTCAGTCTTCC-11,P01_V02_Skin_L,P01,sarcoidosis,Skin_L,None,False,0.026748,2274,5721.0,4.929208,6.100332,True,24,0,1,Tcell,11,Th
AAACGCTTCACGTCCT-11,P01_V02_Skin_L,P01,sarcoidosis,Skin_L,None,False,0.774179,3104,10605.0,6.346063,6.732673,True,24,0,2,other,10,Macro_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GACGGCTCAGCGTCCA-18,4820STDY7389008,s3,normal,dermis,dermal lymphoid,False,7.602468,1320,3319.0,4.218138,20.518228,True,17,0,1,Tcell,14,Treg_SAT1_lo
CATCAGAAGCCACGCT-18,4820STDY7389008,s3,normal,dermis,dermal lymphoid,False,7.602468,1335,3665.0,3.192360,22.319236,True,17,0,1,Tcell,14,Treg_SAT1_lo
TCGGTAAAGGACAGCT-24,4820STDY7389014,s3,normal,epidermis,epidermal lymphoid,False,3.035988,1646,3819.0,0.130924,15.318146,True,23,0,1,Tcell,14,Treg_SAT1_lo
AACTCTTGTCTCGTTC-24,4820STDY7389014,s3,normal,epidermis,epidermal lymphoid,False,3.035988,1375,3240.0,1.913580,19.382716,True,23,0,1,Tcell,14,Treg_SAT1_lo


In [10]:
sampledata = adata[samples.index].copy()

In [11]:
import scanpy as sc

sc.pp.filter_genes(
    sampledata,
    min_cells = 1,
    inplace = True
)

In [10]:
sampledata.to_df().astype(int).to_csv('skin.inflammatory.exp.sample.tsv', sep = '\t')
sampledata.obs.to_csv('skin.inflammatory.meta.sample.tsv', sep = '\t')

In [12]:
adata.to_df().astype(int).to_csv('skin.inflammatory.exp.tsv', sep = '\t')
adata.obs.to_csv('skin.inflammatory.meta.tsv', sep = '\t')

In [1]:
import anndata as ad

tregs = ad.read_h5ad(
    'skin.tregs.h5ad'
)
tregs

AnnData object with n_obs × n_vars = 9773 × 20912
    obs: 'sample_id', 'patient_id', 'status', 'tissue', 'cell_fraction', 'doublet', 'doublet_score', 'nFeature_RNA', 'nCount_RNA', 'percent_mt', 'percent_ribo', 'qc_pass', '_scvi_batch', '_scvi_labels', 'leiden_scvi_0.1', 'coarse_cell_types', 'leiden_scvi_0.4', 'dataset', 'sat1_status', 'clustering', 'sat1_status_majority_vote'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'leiden', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scvi', 'X_umap'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [2]:
tregs = tregs.raw.to_adata()

In [3]:
import scanpy as sc

sc.pp.filter_genes(
    tregs,
    min_cells = 1,
    inplace = True
)

/lisc/data/scratch/menche/daniel/conda/envs/scnova/lib/python3.13/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [4]:
tregs.to_df().astype(int).to_csv('skin.tregs.exp.tsv', sep = '\t')
tregs.obs.to_csv('skin.tregs.meta.tsv', sep = '\t')

In [5]:
sc.pp.filter_genes(
    tregs,
    min_cells = len(tregs) * 0.01,
    inplace = True
)
tregs.to_df().astype(int).to_csv('skin.tregs.exp.filtered.tsv', sep = '\t')